In [0]:
%python
!pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
%python
# import os
# import json
# import pandas as pd
# import numpy as np
# from pyspark.sql.types import *
# from pyspark.sql.functions import *

vector_search_endpoint = 'endpoint_for_vsc_ai2606'
vector_search_index_name= 'ai2605.ai.vs_index_pyspark'
embedding_source_column='chunked_text'
primary_key='chunk_id'
source_table_name = 'workspace.default.tbl_chunked_files'



from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()

try:
    endpoint = vsc.get_endpoint(name=vector_search_endpoint)
    print(f"Endpoint '{vector_search_endpoint}' exists and is {endpoint.get('status', {}).get('state')}.")
except Exception as e:
    print(f"Endpoint '{vector_search_endpoint}' does not exist. Creating it now...")
    vsc.create_endpoint(name=vector_search_endpoint, endpoint_type="STANDARD")
    endpoint = vsc.get_endpoint(name=vector_search_endpoint)
    print(f"Endpoint '{vector_search_endpoint}' created and is {endpoint.get('status', {}).get('state')}.")

In [0]:
%python
vsc.create_delta_sync_index_and_wait(
    endpoint_name = vector_search_endpoint,
    index_name = vector_search_index_name,
    primary_key = primary_key,
    source_table_name = source_table_name,
    embedding_source_column = embedding_source_column,
    pipeline_type='triggered',
    embedding_model_endpoint_name ='databricks-qwen3-embedding-0-6b')

In [0]:
%python
import os
from databricks.vector_search.client import VectorSearchClient


workspace_url = os.environ.get("WORKSPACE_URL")
sp_client_id = os.environ.get("SP_CLIENT_ID")
sp_client_secret = os.environ.get("SP_CLIENT_SECRET")

vsc = VectorSearchClient(
    workspace_url=workspace_url,
    service_principal_client_id=sp_client_id,
    service_principal_client_secret=sp_client_secret
)

index = vsc.get_index(endpoint_name="20260718_aisearch", index_name="workspace.default.tbl_chunked_files_index")

index.similarity_search(num_results=3, columns=["chunked_text","path"], query_text="example_query", query_type="HYBRID")

# filter